# Isolation Forest: priorización de anomalías de consumo

El modelo aprende el comportamiento del alimentador, que se asume mayoritariamente normal. Luego evalúa si los hurtos históricos se ven anómalos frente a dicho comportamiento.

In [27]:
%pip install pandas openpyxl numpy scikit-learn
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import OneClassSVM

HISTORICO_PATH = Path('datos/DATA_HISTORICO_CNR.xlsx')
ALIMENTADOR_PATH = Path('datos/DATA_ALIMENTADOR.xlsx')
RANDOM_STATE = 42
MESES = {'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 'MAYO': 5, 'JUNIO': 6, 'JULIO': 7, 'AGOSTO': 8, 'SETIEMBRE': 9, 'SEPTIEMBRE': 9, 'OCTUBRE': 10, 'NOVIEMBRE': 11, 'DICIEMBRE': 12}


Note: you may need to restart the kernel to use updated packages.


In [28]:
def normalizar(valor):
    valor = unicodedata.normalize('NFKD', str(valor)).encode('ASCII', 'ignore').decode('ASCII')
    return re.sub(r'[^A-Z0-9]+', '_', valor.upper()).strip('_')

def buscar_columna(columnas, *terminos):
    for termino in terminos:
        termino = normalizar(termino)
        for columna in columnas:
            if termino in normalizar(columna):
                return columna
    return None

def columnas_mensuales(columnas, prefijo):
    encontradas = []
    for columna in columnas:
        nombre = normalizar(columna)
        if normalizar(prefijo) in nombre:
            for mes, orden in MESES.items():
                if mes in nombre:
                    encontradas.append((orden, columna))
                    break
    return [columna for _, columna in sorted(encontradas)]

def pendiente(valores):
    valores = np.asarray(valores, dtype=float)
    validos = np.isfinite(valores)
    return np.polyfit(np.arange(len(valores))[validos], valores[validos], 1)[0] if validos.sum() >= 2 else np.nan

def construir_features(raw):
    consumo_cols = columnas_mensuales(raw.columns, 'CONSUMO')
    dias_cols = columnas_mensuales(raw.columns, 'DIA')
    if not consumo_cols:
        raise ValueError('No se encontraron columnas mensuales de consumo.')
    consumo = raw[consumo_cols].apply(pd.to_numeric, errors='coerce')
    if len(dias_cols) == len(consumo_cols):
        dias = raw[dias_cols].apply(pd.to_numeric, errors='coerce').replace(0, np.nan)
        dias.columns = consumo.columns
        diario = consumo.div(dias)
    else:
        diario = consumo / 30.0
    x = pd.DataFrame(index=raw.index)
    x['daily_mean'] = diario.mean(axis=1)
    x['daily_median'] = diario.median(axis=1)
    x['daily_std'] = diario.std(axis=1)
    x['daily_min'] = diario.min(axis=1)
    x['daily_max'] = diario.max(axis=1)
    x['coefficient_variation'] = x['daily_std'] / x['daily_mean'].replace(0, np.nan)
    x['monthly_trend'] = diario.apply(pendiente, axis=1)
    x['recent_3m_daily_mean'] = diario.iloc[:, -3:].mean(axis=1)
    x['recent_vs_annual'] = x['recent_3m_daily_mean'] / x['daily_mean'].replace(0, np.nan)
    x['zero_month_ratio'] = (consumo.fillna(0) == 0).mean(axis=1)
    x['missing_month_ratio'] = consumo.isna().mean(axis=1)
    potencia = buscar_columna(raw.columns, 'POTENCIA_CONTRATADA', 'POTENCIA')
    acometida = buscar_columna(raw.columns, 'ACOMETIDA', 'ACOMET')
    giro = buscar_columna(raw.columns, 'GIRO_COMERCIAL', 'GIRO')
    x['contracted_power_kw'] = pd.to_numeric(raw[potencia], errors='coerce') if potencia else np.nan
    x['connection_type'] = raw[acometida].fillna('NO_DISPONIBLE').astype(str) if acometida else 'NO_DISPONIBLE'
    x['business_type'] = raw[giro].fillna('NO_DISPONIBLE').astype(str) if giro else 'NO_DISPONIBLE'
    return x


In [29]:
historico_raw = pd.read_excel(HISTORICO_PATH)
alimentador_raw = pd.read_excel(ALIMENTADOR_PATH)
historico_x = construir_features(historico_raw)
alimentador_x = construir_features(alimentador_raw).reindex(columns=historico_x.columns)

# Isolation Forest se entrena solo con 80% del alimentador.
alimentador_train, alimentador_holdout = train_test_split(alimentador_x, test_size=0.20, random_state=RANDOM_STATE)
numericas = historico_x.select_dtypes(include='number').columns.tolist()
categoricas = [col for col in historico_x.columns if col not in numericas]
preprocesador = ColumnTransformer([
    ('numericas', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numericas),
    ('categoricas', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categoricas),
])
matriz_train = preprocesador.fit_transform(alimentador_train)
matriz_holdout = preprocesador.transform(alimentador_holdout)
matriz_historico = preprocesador.transform(historico_x)

modelo = IsolationForest(n_estimators=500, contamination=0.02, random_state=RANDOM_STATE, n_jobs=-1).fit(matriz_train)
umbral = float(np.quantile(modelo.score_samples(matriz_train), 0.02))
scores_holdout = modelo.score_samples(matriz_holdout)
scores_historico = modelo.score_samples(matriz_historico)
anomalo_holdout = scores_holdout <= umbral
detectado_historico = scores_historico <= umbral

evaluacion = pd.DataFrame([{
    'Alimentador para entrenar (80%)': len(alimentador_train),
    'Alimentador holdout (20%)': len(alimentador_holdout),
    'Anomalías en holdout alimentador': int(anomalo_holdout.sum()),
    'Históricos confirmados evaluados': len(historico_x),
    'Históricos detectados como anomalía': int(detectado_historico.sum()),
    'Históricos no detectados': int((~detectado_historico).sum()),
    'Cobertura de históricos': f'{detectado_historico.mean():.1%}',
    'Umbral de anomalía': umbral,
}])
display(evaluacion)


,Alimentador para entrenar (80%),Alimentador holdout (20%),Anomalías en holdout alimentador,Históricos confirmados evaluados,Históricos detectados como anomalía,Históricos no detectados,Cobertura de históricos,Umbral de anomalía
0,11960,2991,70,4659,1133,3526,24.3%,-0.602163


## Comparación de umbrales

El porcentaje es la fracción del alimentador considerada anómala. Elija un punto que aumente la cobertura histórica sin producir una lista de inspección inmanejable.

In [30]:
umbrales_a_probar = [0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20, 0.25, 0.28,0.29, 0.30]
scores_train = modelo.score_samples(matriz_train)
filas = []
for proporcion in umbrales_a_probar:
    # El corte se calcula solo con el 80% del alimentador usado para entrenar.
    corte = float(np.quantile(scores_train, proporcion))
    seleccion_holdout = scores_holdout <= corte
    seleccion_historico = scores_historico <= corte
    filas.append({
        'Umbral alimentador': f'{proporcion:.0%}',
        'Suministros estimados a inspeccionar': int(np.ceil(len(alimentador_x) * proporcion)),
        'Anomalías en holdout alimentador': int(seleccion_holdout.sum()),
        'Históricos detectados': int(seleccion_historico.sum()),
        'Históricos no detectados': int((~seleccion_historico).sum()),
        'Cobertura histórica': f'{seleccion_historico.mean():.1%}',
        'Valor de corte': corte,
    })
comparacion_umbrales = pd.DataFrame(filas)
display(comparacion_umbrales)


,Umbral alimentador,Suministros estimados a inspeccionar,Anomalías en holdout alimentador,Históricos detectados,Históricos no detectados,Cobertura histórica,Valor de corte
0,1%,150,43,701,3958,15.0%,-0.664620
1,2%,300,70,1133,3526,24.3%,-0.602163
2,3%,449,105,1424,3235,30.6%,-0.573743
3,5%,748,157,1858,2801,39.9%,-0.537535
4,10%,1496,300,2733,1926,58.7%,-0.488256
5,15%,2243,439,3461,1198,74.3%,-0.458353
6,20%,2991,562,4083,576,87.6%,-0.437815
7,25%,3738,715,4608,51,98.9%,-0.419330
8,28%,4187,807,4657,2,100.0%,-0.412610
9,29%,4336,836,4658,1,100.0%,-0.409644


## Comparación separada de modelos de anomalía

Los tres modelos usan exactamente el mismo 80% del alimentador para aprender normalidad. El histórico permanece fuera del entrenamiento y se utiliza solo para medir cobertura.

In [31]:
# Isolation Forest ya fue entrenado en la celda anterior.
modelos_scores = {
    'Isolation Forest': (modelo.score_samples(matriz_train), modelo.score_samples(matriz_historico)),
}

print('Entrenando LOF...')
lof = LocalOutlierFactor(n_neighbors=35, novelty=True, contamination='auto', n_jobs=-1)
lof.fit(matriz_train)
modelos_scores['Local Outlier Factor'] = (lof.score_samples(matriz_train), lof.score_samples(matriz_historico))

print('Entrenando One-Class SVM...')
ocsvm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
ocsvm.fit(matriz_train)
modelos_scores['One-Class SVM'] = (ocsvm.decision_function(matriz_train).ravel(), ocsvm.decision_function(matriz_historico).ravel())

cortes = [0.01, 0.03, 0.05, 0.10]
filas_modelos = []
for nombre, (scores_train_modelo, scores_historico_modelo) in modelos_scores.items():
    for proporcion in cortes:
        corte = float(np.quantile(scores_train_modelo, proporcion))
        detectados = scores_historico_modelo <= corte
        filas_modelos.append({
            'Modelo': nombre,
            'Lista prioritaria': f'Top {proporcion:.0%}',
            'Suministros a inspeccionar': int(np.ceil(len(alimentador_x) * proporcion)),
            'Históricos detectados': int(detectados.sum()),
            'Históricos no detectados': int((~detectados).sum()),
            'Cobertura histórica': f'{detectados.mean():.1%}',
        })
comparacion_modelos = pd.DataFrame(filas_modelos)
display(comparacion_modelos)
print('El mejor modelo para cada tamaño de lista es el de mayor cobertura histórica.')


Entrenando LOF...
Entrenando One-Class SVM...


,Modelo,Lista prioritaria,Suministros a inspeccionar,Históricos detectados,Históricos no detectados,Cobertura histórica
0,Isolation Forest,Top 1%,150,701,3958,15.0%
1,Isolation Forest,Top 3%,449,1424,3235,30.6%
2,Isolation Forest,Top 5%,748,1858,2801,39.9%
3,Isolation Forest,Top 10%,1496,2733,1926,58.7%
4,Local Outlier Factor,Top 1%,150,294,4365,6.3%
5,Local Outlier Factor,Top 3%,449,3464,1195,74.4%
6,Local Outlier Factor,Top 5%,748,4248,411,91.2%
7,Local Outlier Factor,Top 10%,1496,4524,135,97.1%
8,One-Class SVM,Top 1%,150,637,4022,13.7%
9,One-Class SVM,Top 3%,449,1593,3066,34.2%


El mejor modelo para cada tamaño de lista es el de mayor cobertura histórica.
